# 03 - Model Training and Selection

Huấn luyện và so sánh các mô hình dự đoán giá nhà

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

sys.path.insert(0, os.path.abspath('../src'))

from data.load_data import load_processed_data
from models.train_model import ModelTrainer
from models.evaluate import ModelEvaluator

## 2. Load Processed Data

In [ ]:
# Load processed data
processed_data_path = '../data/processed/house_prices_processed.csv'
df = load_processed_data(processed_data_path)
print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

## 3. Prepare Data for Modeling

In [ ]:
# Assuming 'price' is the target variable
if 'price' in df.columns:
    target_col = 'price'
else:
    # Use the last column as target if 'price' doesn't exist
    target_col = df.columns[-1]

X = df.drop(columns=[target_col])
y = df[target_col]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

## 4. Train Models

In [ ]:
# Initialize trainer and prepare data
trainer = ModelTrainer(random_state=42)
trainer.prepare_data(X, y, test_size=0.2)

In [ ]:
# Train Linear Regression
trainer.train_linear_regression()
y_pred_lr = trainer.model.predict(trainer.X_test)
metrics_lr = ModelEvaluator.print_evaluation_report(trainer.y_test, y_pred_lr, "Linear Regression")

In [ ]:
# Train Ridge Regression
trainer.train_ridge_regression(alpha=1.0)
y_pred_ridge = trainer.model.predict(trainer.X_test)
metrics_ridge = ModelEvaluator.print_evaluation_report(trainer.y_test, y_pred_ridge, "Ridge Regression")

In [ ]:
# Train Random Forest
trainer.train_random_forest(n_estimators=100, max_depth=10)
y_pred_rf = trainer.model.predict(trainer.X_test)
metrics_rf = ModelEvaluator.print_evaluation_report(trainer.y_test, y_pred_rf, "Random Forest")

In [ ]:
# Train Gradient Boosting
trainer.train_gradient_boosting(n_estimators=100, learning_rate=0.1)
y_pred_gb = trainer.model.predict(trainer.X_test)
metrics_gb = ModelEvaluator.print_evaluation_report(trainer.y_test, y_pred_gb, "Gradient Boosting")

## 5. Compare Models

In [ ]:
# Create comparison dataframe
comparison = pd.DataFrame({
    'Linear Regression': metrics_lr,
    'Ridge Regression': metrics_ridge,
    'Random Forest': metrics_rf,
    'Gradient Boosting': metrics_gb
})

print("\nModel Comparison:")
print(comparison)

# Save comparison
comparison.to_csv('../reports/model_comparison.csv')

## 6. Visualize Model Performance

In [ ]:
# Plot R2 scores
fig, ax = plt.subplots(figsize=(10, 6))
comparison.loc['R2'].plot(kind='bar', ax=ax, color=['blue', 'green', 'red', 'purple'])
ax.set_title('Model R² Score Comparison')
ax.set_ylabel('R² Score')
ax.set_xlabel('Models')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/figures/model_r2_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot RMSE scores
fig, ax = plt.subplots(figsize=(10, 6))
comparison.loc['RMSE'].plot(kind='bar', ax=ax, color=['blue', 'green', 'red', 'purple'])
ax.set_title('Model RMSE Score Comparison')
ax.set_ylabel('RMSE')
ax.set_xlabel('Models')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/figures/model_rmse_comparison.png', dpi=300, bbox_inches='tight')
plt.show()